# 1. Gravação de Áudio Com Python (e Uma Pitada de JavaScript) 🎤

In [2]:
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 5 segundos)
print('Ouvindo... pode falar\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

Ouvindo...



<IPython.core.display.Javascript object>

# 2. Reconhecimento de Fala com Whisper (OpenAI) 🧠

In [3]:
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [8]:
import whisper

# Selecione o modelo do Whisper que melhor atenda às suas necessidades:
# https://github.com/openai/whisper#available-models-and-languages
model = whisper.load_model("medium")

result = model.transcribe(record_file, fp16=False)
transcription = result["text"]
detected_language = result["language"]

if detected_language == "pt":
    target_language = "en"
  elif detected_language == "en":
    target_language = "pt"
else:
    target_language = "en"

translated_text = GoogleTranslator(
    source='auto',
    target=target_language
).translate(transcription)

print(f"Texto: {transcription}")
print(f"Idioma detectado: {detected_language}")

Texto:  Olá, tudo bem com você? Como você está?
Idioma detectado: pt


# 3. Tradução Automática 💬

In [9]:
!pip install deep-translator

In [10]:
from deep_translator import GoogleTranslator

# Exemplo de teste (opcional)
translated_text = GoogleTranslator(
    source='auto',
    target=target_language
).translate(transcription)

print(f"Tradução: {translated_text}")

Tradução: Hello, how are you? How are you?


# 4. Sintetizando a Tradução Como Voz 🔊

In [11]:
!pip install gTTS

In [12]:
# 4. Sintetizando a Tradução Como Voz 🔊

from gtts import gTTS
from IPython.display import Audio, display

idioma_map = {
    "pt": "pt",
    "en": "en",
    "es": "es"
}

tts_lang = idioma_map.get(target_language, "en")

gtts_object = gTTS(
    text=translated_text,
    lang=tts_lang,
    slow=False
)

response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)

display(Audio(response_audio, autoplay=True))